# ISE 457 PROJECT REPORT
## Time Series Analysis – Cafes, Restaurants and Catering Services
This notebook includes all original code blocks and explanatory comments from the report.


In [ ]:
# -----------------------------
# Installing and Importing Required Libraries
# These libraries are necessary for time series analysis,
# data manipulation, modeling, and visualization.
# -----------------------------
install.packages("fpp3")
install.packages("urca")

library(tidyverse)
library(tsibble)
library(urca)
library(fpp3)


## Importing Data
We convert the dataset into a time series format using `tsibble`.
The selected series is determined by the student ID.


In [ ]:
# Function to randomly select a valid time series based on student ID
get_my_data2 <- function(student_id) {
  set.seed(student_id)

  all_data <- read.csv("data/all_data_upto202212.csv")
  data_2023to2024 <- read.csv("data/data_202301_202402.csv")

  # Convert to tsibble format
  all_data <- all_data |>
    mutate(Month = yearmonth(Month)) |>
    as_tsibble(index = Month, key = c(State, Industry, Series.ID))

  data_2023to2024 <- data_2023to2024 |>
    mutate(Month = yearmonth(Month)) |>
    as_tsibble(index = Month, key = c(State, Industry, Series.ID))

  # Ensure selected series has no missing values
  while(TRUE) {
    retail <- filter(all_data, Series.ID == sample(Series.ID, 1))
    if(!any(is.na(fill_gaps(retail)$Turnover))) break
  }

  while(TRUE) {
    retail2023to2024 <- filter(data_2023to2024, Series.ID == sample(Series.ID, 1))
    if(!any(is.na(fill_gaps(retail2023to2024)$Turnover))) break
  }

  return(list(retail, retail2023to2024))
}

# Replace with your student ID
retail <- get_my_data2(80703056)[[1]]
retail2023to2024 <- get_my_data2(80703056)[[2]]


## STL Decomposition
STL (Seasonal-Trend decomposition using Loess) decomposes the data into:
- Trend
- Seasonal
- Remainder


In [ ]:
# Apply STL decomposition
retail %>%
  model(STL(Turnover ~ season(window = 9), robust = TRUE)) %>%
  components() %>%
  autoplot() +
  labs(title = "STL Decomposition: Retail Turnover")


## Box-Cox Transformation
Used to stabilize variance in the time series.
Lambda is calculated using the Guerrero method.


In [ ]:
# Estimate optimal lambda
lambda <- retail |>
  features(Turnover, features = guerrero) |>
  pull(lambda_guerrero)

# Plot transformed series
retail |>
  autoplot(box_cox(Turnover, lambda))


## KPSS Unit Root Test
Null Hypothesis (H0): Series is stationary.
If p-value < 0.05 → Reject H0 → Series is non-stationary.


In [ ]:
# Apply KPSS test
retail %>%
  features(Turnover, unitroot_kpss)


## ETS Model
Exponential Smoothing State Space Model


In [ ]:
# Fit ETS model
fit_ets <- retail %>%
  model(ETS(Turnover))

report(fit_ets)

# Forecast 2 years ahead
fit_ets |> forecast(h = "2 years") |> autoplot(retail)


## ARIMA Model
AutoRegressive Integrated Moving Average model applied to Box-Cox transformed data.


In [ ]:
# Fit ARIMA model
fit_arima <- retail |>
  model(ARIMA(box_cox(Turnover, lambda)))

report(fit_arima)

# Forecast 15 months ahead
fit_arima |> forecast(h = "15 months") |> autoplot(retail)
